In [1]:
!git clone https://github.com/rashibharti28/BERT-Quantization-PTQ-QAT-on-dair-ai-emotion.git

Cloning into 'BERT-Quantization-PTQ-QAT-on-dair-ai-emotion'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 43 (delta 13), reused 11 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 402.06 KiB | 7.05 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Filtering content: 100% (2/2), 387.73 MiB | 42.82 MiB/s, done.


In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns



In [3]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)



In [5]:
ds = load_dataset("dair-ai/emotion", "split")
test = ds["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [7]:
import torch
import time
import numpy as np
from datasets import load_from_disk
from transformers import AutoTokenizer
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
from transformers.models.distilbert.modeling_distilbert import DistilBertForSequenceClassification

# Allow loading of full DistilBERT classifier object stored inside the .pt file
torch.serialization.add_safe_globals([DistilBertForSequenceClassification])


# 1. LOAD QUANTIZED MODEL (.pt)

ptq_model_path = "/content/BERT-Quantization-PTQ-QAT-on-dair-ai-emotion/PTQ_distilbert/quantized_model.pt"

model = torch.load(ptq_model_path, weights_only=False)
model.eval()

# 2. LOAD TOKENIZER

finetune_model_dir = "/content/BERT-Quantization-PTQ-QAT-on-dair-ai-emotion/finetuning_bert"
tokenizer = AutoTokenizer.from_pretrained(finetune_model_dir)


# 3. LOAD TEST DATA

test_ds = test.map(
    lambda batch: tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    ),
    batched=True
)

test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# -----------------------------
# 4. INFERENCE + LATENCY
# -----------------------------

latencies = []

def run_batch_inference(batch):
    start = time.time()

    with torch.no_grad():
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        ).logits
        preds = torch.argmax(logits, dim=-1)

    end = time.time()
    latencies.append(end - start)

    return {"preds": preds}

predicted_ds = test_ds.map(run_batch_inference, batched=True, batch_size=32)

# -----------------------------
# 5. METRICS
# -----------------------------
true_labels = np.array(predicted_ds["label"])
pred_labels = np.array(predicted_ds["preds"])

accuracy = accuracy_score(true_labels, pred_labels)
macro_f1 = f1_score(true_labels, pred_labels, average="macro")
per_class_f1 = f1_score(true_labels, pred_labels, average=None)
cmatrix = confusion_matrix(true_labels, pred_labels)

# model size
model_size_mb = round((os.path.getsize(ptq_model_path) / (1024 * 1024)), 2)

# latency per sample
avg_latency_ms = (np.mean(latencies) * 1000)


# 6. PRINT RESULTS

print("\n===== TEST RESULTS =====")
print(f"Accuracy:       {accuracy:.4f}")
print(f"Macro F1:       {macro_f1:.4f}")
print(f"Model Size:     {model_size_mb} MB")
print(f"Latency:        {avg_latency_ms:.2f} ms/sample")

print("\nPer-Class F1:")
for i, f1 in enumerate(per_class_f1):
    print(f"  Class {i}: {f1:.4f}")

print("\nConfusion Matrix:")
print(cmatrix)

print("\nFull Classification Report:")
print(classification_report(true_labels, pred_labels))


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


===== TEST RESULTS =====
Accuracy:       0.9205
Macro F1:       0.8742
Model Size:     132.3 MB
Latency:        4749.42 ms/sample

Per-Class F1:
  Class 0: 0.9634
  Class 1: 0.9446
  Class 2: 0.8054
  Class 3: 0.9140
  Class 4: 0.8764
  Class 5: 0.7413

Confusion Matrix:
[[553   5   0  15   8   0]
 [  3 665  18   2   0   7]
 [  0  37 120   2   0   0]
 [  6   5   1 255   8   0]
 [  4   0   0   8 195  17]
 [  1   1   0   1  10  53]]

Full Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.95      0.96       581
           1       0.93      0.96      0.94       695
           2       0.86      0.75      0.81       159
           3       0.90      0.93      0.91       275
           4       0.88      0.87      0.88       224
           5       0.69      0.80      0.74        66

    accuracy                           0.92      2000
   macro avg       0.87      0.88      0.87      2000
weighted avg       0.92      0.92      0.92    